# Chapter 1 — What Is an LLM?

## LLM Learning From Scratch

In this chapter, we will understand:

- What an LLM is
- How language modeling works
- What tokens are
- How next-token prediction works
- Training vs inference
- Parameters and weights
- How a tiny language model can be built from scratch

The goal is to understand the basic idea behind an LLM before learning
tokenization, embeddings, neural networks, and Transformers.

#  AI → ML → Deep Learning → NLP → LLM

Before understanding an LLM, we need to understand where it fits.

### Artificial Intelligence (AI)

AI is the broad field of building systems that perform tasks that
normally require human intelligence.

Examples:

- Understanding language
- Recognizing images
- Making predictions
- Planning
- Decision making

### Machine Learning (ML)

Machine Learning is a subset of AI where systems learn patterns from data
instead of being explicitly programmed with every rule.

### Deep Learning

Deep Learning is a subset of ML that uses neural networks with many
layers to learn complex patterns.

### Natural Language Processing (NLP)

NLP focuses on enabling computers to work with human language.

Examples:

- Translation
- Text classification
- Summarization
- Question answering
- Text generation

### Large Language Model (LLM)

An LLM is a neural network trained on large amounts of text to model
language.

At its core, a language model learns:

> Given the previous tokens, what token is likely to come next?

This simple idea is the foundation of modern text generation.


## The Relationship

AI
└── Machine Learning
    └── Deep Learning
        └── NLP
            └── Large Language Models

#  What Is Language Modeling?

A language model assigns probabilities to sequences of language.

For example:

> I am going to the ___

Possible next words might be:

- market
- office
- school
- store
- gym

A language model estimates how likely each possible continuation is.

Mathematically, we can think of it as:

P(next token | previous tokens)

For example:

P("market" | "I am going to the")

The model learns these probabilities from examples in its training data.

The key idea is:

**Language modeling = predicting what comes next.**

In [1]:
sentence="I am going to the market"


words=sentence.split()

for i in range(len(words)-1):
  context=words[:i+1]
  target=words[i+1]


  print("Context:",context)
  print("Next Word:",target)
  print()

Context: ['I']
Next Word: am

Context: ['I', 'am']
Next Word: going

Context: ['I', 'am', 'going']
Next Word: to

Context: ['I', 'am', 'going', 'to']
Next Word: the

Context: ['I', 'am', 'going', 'to', 'the']
Next Word: market



In [6]:
text="I love Python"

tokens=text.split()
vocabulary=sorted(set(tokens))

token_to_id={
  token:i
  for i, token in enumerate(vocabulary)
}

print("Token:",tokens)
print("Vocabulary:",vocabulary)
print("Token IDs",token_to_id)

Token: ['I', 'love', 'Python']
Vocabulary: ['I', 'Python', 'love']
Token IDs {'I': 0, 'Python': 1, 'love': 2}


# Next-Token Prediction

Suppose our vocabulary is:

["I", "am", "happy"]

The model receives:

"I am"

and needs to predict:

"happy"

But internally, the model does not simply say:

> The answer is "happy".

It produces a score for every possible token.

For example:

I       → 0.02
am      → 0.01
happy   → 0.90
sad     → 0.05
today   → 0.02

These values represent the model's predicted probabilities.

The token with the highest probability is the model's preferred prediction.

In [7]:
vocabulary = ["I", "am", "happy", "sad", "today"]

probabilities = {
    "I": 0.02,
    "am": 0.01,
    "happy": 0.90,
    "sad": 0.05,
    "today": 0.02
}

for token, probability in probabilities.items():
    print(f"{token:>6} -> {probability}")

     I -> 0.02
    am -> 0.01
 happy -> 0.9
   sad -> 0.05
 today -> 0.02


In [9]:
prediction=max(probabilities,key=probabilities.get)

print("Predicted next token:",prediction)

Predicted next token: happy


#  Training vs Inference

There are two major phases.

## Training

During training, the model sees examples of language and adjusts its
parameters so that its predictions become better.

Conceptually:

Input:

"I am going to the"

Target:

"market"

The model predicts something.

If the prediction is wrong, we calculate how wrong it was and update
the model's parameters.

This happens repeatedly across a huge amount of training data.

---

## Inference

Inference happens after the model has been trained.

We give it a prompt:

"I am going to the"

The model predicts a next token:

"market"

Then that token is added to the input.

The model predicts another token.

This continues one token at a time.

```text
"I am going to the"
          ↓
      "market"
          ↓
"I am going to the market"
          ↓
       "today"
          ↓
"I am going to the market today"

In [12]:
import numpy as np

weights=np.array([
    0.2,
    -0.5,
    0.8,
    0.1
])

print("Weights:",weights)

Weights: [ 0.2 -0.5  0.8  0.1]


#  A Tiny Language Model

Now let's combine the ideas.

We will build a very small language model that learns which word usually
comes after another word.

We will use a tiny dataset:

"the cat sleeps"
"the cat eats"
"the dog sleeps"
"the dog eats"

From this data, the model can learn relationships such as:

the → cat / dog
cat → sleeps / eats
dog → sleeps / eats

This is NOT how modern LLMs are implemented.

It is intentionally simple so that we can see the core idea clearly.

In [13]:
text="""
the cat sleeps
the cat eats
the dog sleeps
the dog eats
"""

words=text.split()

print(words)

['the', 'cat', 'sleeps', 'the', 'cat', 'eats', 'the', 'dog', 'sleeps', 'the', 'dog', 'eats']


In [14]:
pairs=[]

for i in range(len(words)-1):
  current_word=words[i]
  next_word=words[i+1]

  pairs.append((current_word, next_word))

for current,next_word in pairs:
  print(f"{current}->{next_word}")

the->cat
cat->sleeps
sleeps->the
the->cat
cat->eats
eats->the
the->dog
dog->sleeps
sleeps->the
the->dog
dog->eats


#  Learning Next-Word Probabilities

Instead of manually specifying probabilities, we can count how frequently
each word follows another word.

For example, if our data contains:

the → cat
the → cat
the → dog

then:

P(cat | the) = 2/3

P(dog | the) = 1/3

The model can use these learned probabilities to predict the next word.

In [15]:
from collections import defaultdict, Counter

next_word_counts=defaultdict(Counter)

for current_word, next_word in pairs:
  next_word_counts[current_word][next_word]+=1

for current_word,counts in next_word_counts.items():
  print(current_word, "->",dict(counts))

the -> {'cat': 2, 'dog': 2}
cat -> {'sleeps': 1, 'eats': 1}
sleeps -> {'the': 2}
eats -> {'the': 1}
dog -> {'sleeps': 1, 'eats': 1}


In [16]:
next_word_probabilities = {}

for current_word, counts in next_word_counts.items():
    total = sum(counts.values())

    next_word_probabilities[current_word] = {
        word: count / total
        for word, count in counts.items()
    }

next_word_probabilities

{'the': {'cat': 0.5, 'dog': 0.5},
 'cat': {'sleeps': 0.5, 'eats': 0.5},
 'sleeps': {'the': 1.0},
 'eats': {'the': 1.0},
 'dog': {'sleeps': 0.5, 'eats': 0.5}}

In [17]:
import random

def predict_next_word(current_word):
  probabilities=next_word_probabilities[current_word]

  words=list(probabilities.keys())
  probs=list(probabilities.values())

  return random.choices(words,weights=probs,k=1)[0]


current_word="the"


for _ in range(5):
  next_word=predict_next_word(current_word)
  print(current_word, "->", next_word)
  current_word=next_word


the -> cat
cat -> sleeps
sleeps -> the
the -> cat
cat -> eats


 What We Learned

The fundamental loop behind language generation is:

```text
Text
 ↓
Tokens
 ↓
Model
 ↓
Probability distribution
 ↓
Next-token prediction
 ↓
Add predicted token
 ↓
Repeat



Training data
     ↓
Input + target
     ↓
Model prediction
     ↓
Loss
     ↓
Parameter update
     ↓
Better model



Prompt
 ↓
Model
 ↓
Next-token probabilities
 ↓
Choose token
 ↓
Add token to context
 ↓
Repeat

